In [1]:
! git init

Initialized empty Git repository in C:/Users/casus/OneDrive/Desktop/AERO ORGANISED/MY WORK, PROJECTS ETC/RESEARCH & PATENTS/ANUP SIR/Optimising_conference_data/.git/


In [2]:
! git add . 

In [3]:
! git commit -m "Using calculated suction_location columns "

[main 85cc014] Using calculated suction_location columns
 7 files changed, 1307 insertions(+), 320 deletions(-)
 create mode 100644 calc_suction_optimised.csv
 create mode 100644 suction_only_merged_fields.csv


In [4]:
! git push

To https://github.com/HendrichMugisha/anup_sir-conference-optimisation.git
   cfee41b..85cc014  main -> main


In [6]:
! git remote add origin https://github.com/HendrichMugisha/anup_sir-conference-optimisation.git
! git branch -M main
! git push -u origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/HendrichMugisha/anup_sir-conference-optimisation.git
 * [new branch]      main -> main


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

In [3]:
!dir

 Volume in drive C is OS
 Volume Serial Number is 1E87-BAE1

 Directory of C:\Users\casus\OneDrive\Desktop\AERO ORGANISED\MY WORK, PROJECTS ETC\RESEARCH & PATENTS\ANUP SIR\Optimising_conference_data

07/10/2025  15:19    <DIR>          .
07/10/2025  14:33    <DIR>          ..
07/10/2025  14:36    <DIR>          .ipynb_checkpoints
07/10/2025  14:33            16,685 CL & CD Graphs of Injection & Suction.xlsx
07/10/2025  15:00               941 conference_injection_only.csv
07/10/2025  15:16               941 conference_suction_only.csv
07/10/2025  15:06            14,539 injection_only.csv
07/10/2025  14:38               617 Optimising_conference_work.ipynb
06/10/2025  11:09           457,665 Performance_improvement_of_horizontal_axis_wind_turbine_using_active_control_methods.pdf
07/10/2025  15:19            14,972 suction_only.csv
               7 File(s)        506,360 bytes
               3 Dir(s)  190,766,604,288 bytes free


In [12]:
injection_df = pd.read_csv('./injection_only.csv')
injection_df.columns = ['AOA', 'injection_location', 'injection_slot_size', 'cl', 'cd']
injection_df

,AOA,injection_location,injection_slot_size,cl,cd
0,-20.0,4,0.5,-0.65,0.280
1,-19.5,4,0.5,-0.66,0.265
2,-19.0,4,0.5,-0.67,0.250
3,-18.5,4,0.5,-0.68,0.235
4,-18.0,4,0.5,-0.69,0.220
...,...,...,...,...,...
679,25.5,50,2.0,1.77,1.052
680,26.0,50,2.0,1.74,1.084
681,26.5,50,2.0,1.71,1.116
682,27.0,50,2.0,1.68,1.148


In [18]:
# setting up the data
data_df = injection_df.drop(columns=['cl', 'cd'])
data_df
injection_data = np.array(data_df)
injection_data

array([[-20. ,   4. ,   0.5],
       [-19.5,   4. ,   0.5],
       [-19. ,   4. ,   0.5],
       ...,
       [ 26.5,  50. ,   2. ],
       [ 27. ,  50. ,   2. ],
       [ 27.5,  50. ,   2. ]], shape=(684, 3))

In [20]:
#setting up the targets
injection_cl = np.array(injection_df['cl'])
injection_cd = np.array(injection_df['cd'])
# injection_cl

In [23]:
# splitting the data for cl and and cd separately
from sklearn.model_selection import train_test_split

X_train_cl, X_test_cl, y_train_cl, y_test_cl = train_test_split(injection_data, injection_cl, random_state=0)
X_train_cd, X_test_cd, y_train_cd, y_test_cd = train_test_split(injection_data, injection_cd, random_state=0)

# X_train_cd
# X_train_cl

array([[ 21.5 ,   2.  ,   0.75],
       [ 13.  ,   4.  ,   0.5 ],
       [ 22.  ,   4.  ,   0.5 ],
       ...,
       [-15.  ,   4.  ,   0.5 ],
       [  0.5 ,  50.  ,   2.  ],
       [ 21.5 ,   4.  ,   0.5 ]], shape=(513, 3))

### Data Scaling

In [27]:
# scaling the data
from sklearn.preprocessing import StandardScaler

scaler_cl = StandardScaler().fit(X_train_cl)
scaler_cd = StandardScaler().fit(X_train_cd)
# scaling the cl data

X_train_cl_scaled = scaler_cl.transform(X_train_cl)
X_test_cl_scaled = scaler_cl.transform(X_test_cd)

# scaling the cd data
X_train_cd_scaled = scaler_cd.transform(X_train_cd)
X_test_cd_scaled = scaler_cd.transform(X_test_cd)

# X_train_cd_scaled
# X_train_cl_scaled


### Training the models

In [36]:
from sklearn.neural_network import MLPRegressor

mlp_cl = MLPRegressor(random_state=0, early_stopping=True, max_iter=300).fit(X_train_cl_scaled, y_train_cl)

mlp_cd = MLPRegressor(random_state=0, early_stopping=True, max_iter=250).fit(X_train_cd_scaled, y_train_cd)

print("CL prediction accuracy")
print(f"Cl training score: {mlp_cl.score(X_train_cl_scaled, y_train_cl)}")
print(f"Cl Test score: {mlp_cl.score(X_test_cl_scaled, y_test_cl)}")


print("\nCD prediction accuracy")
print(f"Cd training score: {mlp_cd.score(X_train_cd_scaled, y_train_cd)}")
print(f"Cd Test score: {mlp_cd.score(X_test_cd_scaled, y_test_cd)}")



CL prediction accuracy
Cl training score: 0.9905785212449099
Cl Test score: 0.9910731710875994

CD prediction accuracy
Cd training score: 0.9955304671231388
Cd Test score: 0.9940859543331426


###  Optimising CL

In [50]:
angles = np.arange(0, 28, 0.5)
angles = np.arange(0, 5, 1)
# angles
optimised_cl_values = []
optimised_injection_locations= []
optimised_injection_slot_sizes= []

In [52]:
import skopt
from skopt.optimizer import gp_minimize
from skopt.space import Real, Integer, Categorical

for angle in angles:
    print(f'optimising for {angle} deg', end=',')
    def objective_function_cl(params):
        aoa = angle
        injection_location, injection_slot_size = params
        X_input = [[aoa, injection_location, injection_slot_size]]
        X_input_scaled = scaler_cl.transform(X_input)
        predicted_cl = mlp_cl.predict(X_input_scaled)


        return -predicted_cl[0]

    space = [
        Integer(2, 50, name='injection_location'),
        Real(0.5, 2, name='injection_slot_size')
    ]

    result = gp_minimize(
        func=objective_function_cl,
        dimensions=space,
        n_calls = 30,
        random_state = 0
    )
    optimised_cl_values.append(-result.fun)
    optimised_injection_locations.append(result.x[0])
    optimised_injection_slot_sizes.append(result.x[1])

    print(f' {angle} done!')
    

optimising for 0 deg, 0 done!
optimising for 1 deg, 1 done!
optimising for 2 deg, 2 done!
optimising for 3 deg, 3 done!
optimising for 4 deg, 4 done!


In [54]:
output_dict = {
    'aoa' : angles,
    'optimal_injection_loc' : optimised_injection_locations,
    'optimal_injection_slot_size' : optimised_injection_slot_sizes
}
output_dict

{'aoa': array([0, 1, 2, 3, 4]),
 'optimal_injection_loc': [np.int64(2),
  np.int64(2),
  np.int64(2),
  np.int64(2),
  np.int64(2),
  np.int64(2)],
 'optimal_injection_slot_size': [2.0, 2.0, 2.0, 2.0, 2.0]}